# Four Tickets, Two Different Totals

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_OER/blob/main/course_materials/notebooks/09_synthesis_review.ipynb)

[View on GitHub](https://github.com/lolusername/CST4714_OER/blob/main/course_materials/notebooks/09_synthesis_review.ipynb)

The operations dashboard says there are **three active tickets**. Its staff
workload report adds up to **two**. Is a ticket missing, or do these results
describe different populations?

This final review connects SQL, JSON, safe changes, and recovery checks. Each
code cell has a small result you can explain. Run top to bottom before changing
an example. There are no installations, passwords, downloads, or cloud calls.
The synthetic records below have no real personal information.

Python's `sqlite3` package executes real SQL inside this notebook. It does not
connect to Supabase/PostgreSQL or MongoDB Atlas. The local database lasts only
while its connection stays open. This deliberately small example teaches query
meaning, not cloud performance or PostgreSQL's concurrency behavior.

For the final GitHub concept lab, adapt **one** idea or another completed course
example. This notebook is a worked reference, not an additional submission.

## 1. Inspect the Records and the Report

One `staff` row means one staff member. One `tickets` row means one ticket.
`ticket_id` identifies a ticket. `assignee_id` refers to a staff member, or is
SQL `NULL` when the ticket is unassigned. Active means `new`, `open`, or
`in_progress` in this example.

`execute` sends one SQL statement to SQLite. `executescript` runs the fixed
setup statements below. `fetchall` returns result rows as Python tuples.
Running this setup again replaces **only this notebook's in-memory database**.

In [ ]:
import sqlite3
import json

if "db" in globals():
    db.close()
db = sqlite3.connect(":memory:", isolation_level=None)
# Foreign-key checking is a connection setting in SQLite.
db.execute("PRAGMA foreign_keys = ON")
db.executescript("""
CREATE TABLE staff (
    agent_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL
);
CREATE TABLE tickets (
    ticket_id INTEGER PRIMARY KEY,
    assignee_id INTEGER REFERENCES staff(agent_id),
    status TEXT NOT NULL
        CHECK (status IN ('new', 'open', 'in_progress', 'resolved'))
);
INSERT INTO staff VALUES (201, 'Priya'), (202, 'Noah'), (203, 'Elena');
INSERT INTO tickets VALUES
    (1, 201, 'open'), (2, 201, 'resolved'),
    (3, NULL, 'new'), (4, 202, 'open');
""")
print("Staff:", db.execute("SELECT * FROM staff ORDER BY agent_id").fetchall())
print("Tickets:", db.execute("SELECT * FROM tickets ORDER BY ticket_id").fetchall())

### The Active-Ticket Population

Before running the cell, identify the active ticket IDs from the four setup
rows. The SQL filter is selection in relational algebra. Choosing the output
columns is projection. The result still has one row per matching ticket.

In [ ]:
active_sql = """
SELECT ticket_id, assignee_id, status
FROM tickets
WHERE status IN ('new', 'open', 'in_progress')
ORDER BY ticket_id;
"""
active_rows = db.execute(active_sql).fetchall()
print(active_rows)
print("Active tickets:", len(active_rows))

Expected IDs: **1, 3, 4**. Python prints SQL `NULL` as `None`. There are three
active tickets, including the unassigned ticket 3.

### The Staff Report

The left join below keeps every staff member. The status test belongs to the
matching rule in `ON`. It chooses which tickets can contribute to each staff
member's count. `COUNT(t.ticket_id)` counts matched tickets, not the placeholder
row created for someone who has no matching ticket.

In [ ]:
report_sql = """
SELECT s.agent_id, s.name, COUNT(t.ticket_id) AS active_count
FROM staff AS s
LEFT JOIN tickets AS t
    ON t.assignee_id = s.agent_id
   AND t.status IN ('new', 'open', 'in_progress')
GROUP BY s.agent_id, s.name
ORDER BY s.agent_id;
"""
report_rows = db.execute(report_sql).fetchall()
print(report_rows)
print("Assigned active tickets:", sum(row[2] for row in report_rows))

Expected result: **Priya 1, Noah 1, Elena 0**. The counts sum to two because
the report counts assigned active tickets. The left join preserves **staff**,
not all tickets. The unassigned ticket belongs in a separate queue, or another
explicit category, if the dashboard needs the total backlog.

### Two Plausible but Wrong Edits

The first query counts the joined rows with `COUNT(*)`. The second filters
ticket status after the left join. Predict what each does to Elena before
running it. Neither is a correction for the unassigned ticket.

In [ ]:
count_star_sql = report_sql.replace("COUNT(t.ticket_id)", "COUNT(*)")
print("COUNT(*) result:", db.execute(count_star_sql).fetchall())

where_sql = """
SELECT s.agent_id, s.name, COUNT(t.ticket_id) AS active_count
FROM staff AS s
LEFT JOIN tickets AS t ON t.assignee_id = s.agent_id
WHERE t.status IN ('new', 'open', 'in_progress')
GROUP BY s.agent_id, s.name
ORDER BY s.agent_id;
"""
print("Status filter in WHERE:", db.execute(where_sql).fetchall())

`COUNT(*)` gives Elena **1**, even though she has no ticket. Its total happens
to equal the whole backlog of three, but it attributes a nonexistent ticket
to Elena. A matching total can hide a wrong explanation.

The `WHERE` version omits Elena entirely. Her joined ticket fields are NULL,
so the active-status condition is not true. For this requirement, we need her
row with zero.

## 2. Make One Controlled Change

Change `TARGET_ASSIGNEE` to another existing staff ID if you want to test your
explanation. The cell temporarily assigns ticket 3, prints both reports, then
rolls the change back in `finally`, even if reading the report fails. `?` binds
the Python value as data. It is not string concatenation.

With target 203, predict Elena's count and the total number of active tickets.

In [ ]:
TARGET_ASSIGNEE = 203
db.execute("BEGIN")
try:
    db.execute(
        "UPDATE tickets SET assignee_id = ? WHERE ticket_id = ?",
        (TARGET_ASSIGNEE, 3),
    )
    changed_report = db.execute(report_sql).fetchall()
    print("Temporary assignment:", changed_report)
    print("Active tickets:", len(db.execute(active_sql).fetchall()))
finally:
    db.execute("ROLLBACK")
print("After rollback:", db.execute(report_sql).fetchall())

Elena temporarily has one active ticket. The assigned counts now sum to three,
but the active-ticket population remains IDs 1, 3, 4. Assignment changes the
category, not the number of tickets. After rollback, Elena returns to zero.
Rerunning the cell should reproduce the same comparison.

### An Expected Failure and a Transaction Boundary

Here a valid assignment precedes an invalid status. We catch the expected
constraint error and explicitly roll back the **whole transaction**. A failed
SQLite statement alone does not generally undo earlier successful statements.
Predict whether ticket 3 will remain assigned afterward.

In [ ]:
db.execute("BEGIN")
try:
    db.execute("UPDATE tickets SET assignee_id = 203 WHERE ticket_id = 3")
    db.execute("UPDATE tickets SET status = 'finished' WHERE ticket_id = 3")
except sqlite3.IntegrityError as error:
    print("Expected rejection:", error)
finally:
    db.execute("ROLLBACK")
print("Ticket 3:", db.execute(
    "SELECT * FROM tickets WHERE ticket_id = 3"
).fetchone())

Expected ticket 3: **(3, None, 'new')**. The CHECK rule rejects `finished`.
Explicit rollback also discards the earlier assignment. This is an integrity
and transaction example. It does not test user permissions or RLS.

## 3. Explain the Same Facts as Documents

This cell converts query rows to Python dictionaries and then JSON text.
`None` becomes JSON `null`. It creates no MongoDB collection. Each dictionary
still means one ticket, and `assignee_id` remains a reference rather than an
embedded staff profile.

In [ ]:
documents = []
for ticket_id, assignee_id, status in db.execute(
    "SELECT * FROM tickets ORDER BY ticket_id"
):
    documents.append({
        "_id": ticket_id,
        "assignee_id": assignee_id,
        "status": status,
    })
print(json.dumps(documents, indent=2))

active_by_assignee = {}
for document in documents:
    if document["status"] in ("new", "open", "in_progress"):
        assignee = document["assignee_id"]
        active_by_assignee[assignee] = active_by_assignee.get(assignee, 0) + 1
print("Active tickets grouped by assignee:", active_by_assignee)

Expected groups: **201: 1, None: 1, 202: 1**. Elena has no group because the
input contains tickets, not every staff member. The unassigned group is present.
This Python loop illustrates filtering and grouping. It is not an MQL engine.

For comparison, this MongoDB shell pipeline expresses the same ticket grouping
if `tickets` already contains the four documents above. It is a **reference
only** and is not executed by this notebook:

```javascript
db.tickets.aggregate([
  { $match: { status: { $in: ["new", "open", "in_progress"] } } },
  { $group: { _id: "$assignee_id", active_count: { $sum: 1 } } }
])
```

The expected groups are `_id` 201, 202, and null, each with count 1.
Group output order is unspecified here. Producing a zero-inclusive staff
report requires starting with or merging the staff population. Switching query
languages does not remove the need to define what the result should include.

## 4. Compare Values After a Restore

We export this small database as SQL text and load it into a **separate
in-memory connection**. That demonstrates a logical restore mechanism, but the
text is only a Python variable: it is not a durable off-machine backup.

The next cell deliberately changes ticket 1 **only in the restored copy**.
Both databases will still have four tickets. Predict whether comparing the
ordered rows will accept that copy as equal to the source.

In [ ]:
if "restored" in globals():
    restored.close()
dump_sql = "\n".join(db.iterdump())
restored = sqlite3.connect(":memory:", isolation_level=None)
restored.execute("PRAGMA foreign_keys = ON")
restored.executescript(dump_sql)
compare_sql = "SELECT * FROM tickets ORDER BY ticket_id"
expected_rows = db.execute(compare_sql).fetchall()
print("Initial restore matches:", expected_rows == restored.execute(compare_sql).fetchall())

# Only the disposable restored copy receives this intentional wrong value.
restored.execute("UPDATE tickets SET status = 'resolved' WHERE ticket_id = 1")
restored_rows = restored.execute(compare_sql).fetchall()
print("Ticket counts:", len(expected_rows), len(restored_rows))
print("Values match:", expected_rows == restored_rows)
print("Source ticket 1:", expected_rows[0])
print("Restored ticket 1:", restored_rows[0])

The initial restore matches. After the deliberate change, both counts are
**4**, but the value comparison is **False**. The original ticket 1 stays
`open`; the copy says `resolved`. To repeat the restore, rerun its cell. The
cell recreates its own separate target from the unchanged source.

A full PostgreSQL or MongoDB recovery review also considers the appropriate
constraints, indexes, identities, permissions, data types, and service settings.
This SQLite example does not replace those platform-specific checks.

## One Idea for Your GitHub Guide

Choose one result to explain. For example, change the assignment target or
replace a sample row, predict the result, and compare it with your run. Explain
why the output changed. A README with code and a concrete explanation is enough;
a separate notebook file is optional. Follow the Week 15 lab for submission.

A useful opening might be: "This example shows why a staff workload report can
exclude an unassigned ticket even though that ticket is still active." Continue
with your own example, output, explanation, and limit. You do not need to claim
production experience to discuss the query accurately.

## Cleanup

Close both local connections. No cloud service or network access rule was opened.
To run again after cleanup, start at the setup cell.

In [ ]:
restored.close()
db.close()
print("Closed both in-memory databases.")

## Sources and Reuse

- Course textbook, Chapter 15, worked SQL explanation.
- [PostgreSQL table expressions](https://www.postgresql.org/docs/current/queries-table-expressions.html)
- [Python sqlite3 documentation](https://docs.python.org/3/library/sqlite3.html)
- [MongoDB $group](https://www.mongodb.com/docs/manual/reference/operator/aggregation/group/)

Course prose: CC BY-NC-SA 4.0. Original code: MIT. Synthetic records: CC0.